<np>

# PYTORCH

In [2]:
import torch

print(torch.rand(1, 3), torch.__version__, torch.cuda.is_available())

tensor([[0.8961, 0.8759, 0.3115]]) 2.11.0+cu130 True


In [1]:
from torch import nn # modulo de neural networks
from torch.utils.data import DataLoader # wrapper de datasets -> le damos un tamaño de batch, y datalloader nos proporciona los datos en grupos de tamaño batch
from torchvision import datasets
from torchvision.transforms import ToTensor



In [38]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor()) # toTensor() -> nos devuelve en formato tensor
# Download test data from open datasets.
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

In [39]:
batch_size = 64
# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

# N:    C:     H:altura   W:
# 64 dim

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
# Define model
class NeuralNetwork(nn.Module):     # heredamos de nn.Module -> Clase base para redes, implementa gradiente... -> DEBE HEREDAR DE MODULE PARA QUE SEPA QUE ESTO ES UNA RED
    # En  una red se necesitan 2 metodos:    
    def __init__(self):     # Monta la red, la define
        super().__init__()
        self.flatten = nn.Flatten()     # convierte la matriz en un vector
        self.linear_relu_stack = nn.Sequential(     # define la red como una clase secuencial:
            nn.Linear(28*28, 512),                  # 1 lineal (INPUT: 28*28 neuronas OUTPUT: 512)
            nn.ReLU(),                              # LE DA LA NO-LINEALIDAD A LA RED--> NECESARIO
            nn.Linear(512, 512),                    # 1 lineal (512 neuronas * 512)
            nn.ReLU(),
            nn.Linear(512, 10),                   # 1 lineal (512 neuronas * 10)

        )

    def forward(self, x):                   # permite el proceso de la red. Define el orden de las capas
        x = self.flatten(x)                 # recibe una muestra x y se la pasa a flatten(x)
        logits = self.linear_relu_stack(x)  # pasamos dicha muestra a la red, que atraviesa toda la red
        return logits
model = NeuralNetwork().to(device)      # INICIAMOS EL MODELO
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [41]:
# descenso por gradiente necesita el forward, calcular la perdida y realiazr el backwards:
# PERDIDA
loss_fn = nn.CrossEntropyLoss()
# OPTIMIZADOR
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3) # SGD: stochastic gradient descent
# ENTRENAMIENTO
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)          
    model.train()
    for batch, (X, y) in enumerate(dataloader):     # iteramos sobre las distintas muestra de tamaño batch que devuelve DataLoader
        X, y = X.to(device), y.to(device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)         
        # Backpropagation
        loss.backward()         
        optimizer.step()                # actualiza los pesos
        optimizer.zero_grad()           # reseteamos el gradiente a 0
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")


In [42]:
# TEST: estimacion del rendimiento teorico con datos de test
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()                    # ponemos el modelo formato evaluacion -> no aplicar gradientes
    test_loss, correct = 0, 0
    with torch.no_grad():           # asegurarnos que no calcule gradientes, solo realizar forward
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)   # se lo pasamos a la grafica
            pred = model(X)                     # Calculamos la prediccion
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches        # calculamos la perdida con respecto al test
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [43]:
# COMENZAMOS EL ENTRENAMIENTO
# BACKPROP: con un numero de epocas dado:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

# En cada iteracion vemos como va evolucionando la perdida
# Con 1 epoca no ha llegado a apredner mucho (accuracy = 44.9)
# Conforme aumentamos el numero de epocas, aprende mejor
#

Epoch 1
-------------------------------
loss: 2.305527 [   64/60000]
loss: 2.295880 [ 6464/60000]
loss: 2.282445 [12864/60000]
loss: 2.273293 [19264/60000]
loss: 2.250626 [25664/60000]
loss: 2.225941 [32064/60000]
loss: 2.225672 [38464/60000]
loss: 2.196478 [44864/60000]
loss: 2.195328 [51264/60000]
loss: 2.163653 [57664/60000]
Test Error: 
 Accuracy: 32.9%, Avg loss: 2.157666 

Epoch 2
-------------------------------
loss: 2.166967 [   64/60000]
loss: 2.165263 [ 6464/60000]
loss: 2.113254 [12864/60000]
loss: 2.128041 [19264/60000]
loss: 2.078624 [25664/60000]
loss: 2.017238 [32064/60000]
loss: 2.040996 [38464/60000]
loss: 1.968608 [44864/60000]
loss: 1.968465 [51264/60000]
loss: 1.904465 [57664/60000]
Test Error: 
 Accuracy: 55.0%, Avg loss: 1.902919 

Epoch 3
-------------------------------
loss: 1.930759 [   64/60000]
loss: 1.913396 [ 6464/60000]
loss: 1.803162 [12864/60000]
loss: 1.842159 [19264/60000]
loss: 1.732669 [25664/60000]
loss: 1.681356 [32064/60000]
loss: 1.699651 [38464/

In [11]:
# PARA GUARDAR LOS PESOS CALCULADOS:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [12]:
# PODEMOS VOLVER A CARGAR LOS DATOS
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [37]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
model.eval()
x, y = test_data[0][0], test_data[0][1]     # cargamos la primera muestra
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Dress", Actual: "Ankle boot"


<np>

# LEARN THE BASICS: TENSORS

In [26]:
import numpy as np

# Inicializacion directamente a partir de datos:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)
print(x_data)


tensor([[1, 2],
        [3, 4]])


In [27]:
# Inicializacion desde un array NumPy
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(x_np)


tensor([[1, 2],
        [3, 4]])


In [28]:
# Inicializacion desde otro tensor
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")
x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.2295, 0.5500],
        [0.5114, 0.3808]]) 



In [44]:
# Inicializacion con valores constantes o aleatorios
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)
print(f"Random Tensor: \n {rand_tensor}")
print(f"Ones Tensor: \n {ones_tensor}")
print(f"Zeros Tensor: \n {zeros_tensor}")


Random Tensor: 
 tensor([[0.7730, 0.1272, 0.7946],
        [0.8707, 0.5215, 0.5216]])
Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


In [45]:
# Atributos de un tensor
tensor = torch.rand(3,4)
print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


In [46]:
# Movimeinto de un tensor al acelerador (GPU) actual si esta disponible
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())
print(f"Device tensor is stored on: {tensor.device}")

Device tensor is stored on: cuda:0


In [47]:
# indexacion y recorte al estilo numpy
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [ ]:
# Concatenacion de tensor
t1 = torch.cat([tensor, tensor, tensor], dim=1)
print(t1)